# EduHinglish: Inference & Demo on Colab T4 (no training)

Loads the **already trained** model and runs the full pipeline:
**M1** (script detection, normalisation, MuRIL intent) → **M2** (ChromaDB NCERT retrieval) → **M3** (Gemma-3-4B-it + EduHinglish LoRA).

- Model weights come from the Hugging Face Hub (pushed by notebook 05 / `scripts/push_to_hub.py`).
- The knowledge base is built once and cached on Drive, so later sessions start in a few minutes.
- The last cell opens a **public Gradio link** that works from any device. The same link also exposes an
  HTTP API (`gradio_client`) that your frontend/backend can call later.

> **Runtime**: Runtime → Change runtime type → **T4 GPU**. Secrets: `HF_TOKEN` (read), optional `GITHUB_TOKEN` if the repo is private.

In [ ]:
!pip install -q -U "bitsandbytes>=0.43.0" "transformers>=4.47.0" "peft>=0.14.0" "accelerate>=1.0.0" \
    "chromadb>=0.4.24" "sentence-transformers>=2.7.0" nltk colorama gradio

## 1. Settings

In [ ]:
GITHUB_REPO = "https://github.com/PersuasivePost/EduHinglish.git"
BRANCH      = "develop"

HF_ADAPTER_REPO = "your-hf-username/eduhinglish-gemma-3-4b-it-lora"   # Gemma LoRA (notebook 05)
HF_INTENT_REPO  = "your-hf-username/eduhinglish-muril-intent"          # MuRIL intent; None = skip intent hint

BASE_DIR   = "/content/drive/MyDrive/EduHinglish"
KB_CACHE   = f"{BASE_DIR}/knowledge_base_v4"    # cached ChromaDB on Drive
REBUILD_KB = False                              # True after adding chapters to src/chunker.py

## 2. Get the code, secrets and Drive

In [ ]:
import os, sys, shutil, subprocess
from google.colab import drive, userdata

drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

try:
    gh_token = userdata.get("GITHUB_TOKEN")
except Exception:
    gh_token = None
url = GITHUB_REPO.replace("https://", f"https://{gh_token}@") if gh_token else GITHUB_REPO

REPO = "/content/EduHinglish"
if os.path.isdir(f"{REPO}/.git"):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src")
subprocess.run([sys.executable, "scripts/download_nltk_data.py"], check=True)

## 3. NCERT knowledge base (cached on Drive)

In [ ]:
KB_LOCAL = f"{REPO}/src/knowledge_base"

def kb_count(path):
    from retriever import NCERTRetriever
    try:
        return NCERTRetriever(db_path=path).get_collection_stats()["total_documents"]
    except Exception as e:
        print("KB load failed:", e)
        return 0

if not REBUILD_KB and os.path.isdir(KB_CACHE):
    shutil.rmtree(KB_LOCAL, ignore_errors=True)
    shutil.copytree(KB_CACHE, KB_LOCAL)

n = kb_count(KB_LOCAL)
if n == 0:
    print("Building knowledge base (all 5 subjects)...")
    subprocess.run([sys.executable, "scripts/build_knowledge_base.py", "--force"], check=True)
    shutil.rmtree(KB_CACHE, ignore_errors=True)
    shutil.copytree(KB_LOCAL, KB_CACHE)
    n = kb_count(KB_LOCAL)
print(f"Knowledge base ready: {n} chunks")

## 4. Load the pipeline (~3–5 min: Gemma 4-bit on the T4)

In [ ]:
os.environ["EDUHINGLISH_ADAPTER"] = HF_ADAPTER_REPO
if HF_INTENT_REPO:
    os.environ["EDUHINGLISH_INTENT_MODEL"] = HF_INTENT_REPO
os.environ["EDUHINGLISH_KB_PATH"] = KB_LOCAL

from pipeline import EduHinglishPipeline
pipe = EduHinglishPipeline(mode="full", use_intent_model=bool(HF_INTENT_REPO))

## 5. Try it

In [ ]:
SAMPLES = [
    ("Mitochondria ko powerhouse of the cell kyon kehte hain?", "biology"),
    ("Ohm ka law kya hai aur iska formula kya hai?", "physics"),
    ("Acid aur base mein kya difference hai?", "chemistry"),
    ("Arithmetic progression ka nth term kaise nikalte hain?", "mathematics"),
    ("Federalism ke main features kya hain?", "social_science"),
]
for q, subject in SAMPLES:
    r = pipe.answer_query(q, subject=subject)
    print("\n" + "#" * 80 + f"\nQ: {q}\nA: {r['generation']['hinglish_answer']}\n")

## 6. Public demo link (accessible from anywhere)
Prints a `https://xxxx.gradio.live` URL. It stays up while this notebook runs (Gradio share links last up to 72 h;
a Colab session ends earlier). The page footer has **"Use via API"** showing how a backend can call it:

```python
from gradio_client import Client
client = Client("https://xxxx.gradio.live")
answer, sources = client.predict("Ohm ka law kya hai?", "physics", "10", api_name="/answer")
```

In [ ]:
import gradio as gr

def answer(query, subject, class_num):
    r = pipe.answer_query(query, subject=None if subject == "all" else subject,
                          class_num=None if class_num == "all" else class_num)
    intent = r.get("intent") or {}
    srcs = "\n".join(f"- **{s['chapter']}** ({s['subject']}, class {s['class']}) — score {s['score']:.2f}"
                     for s in r["retrieval"])
    meta = f"\n\n_Intent: {intent.get('intent', 'n/a')} · total {r['pipeline_trace'][-1]['time_sec']}s_"
    return r["generation"]["hinglish_answer"], srcs + meta

demo = gr.Interface(
    fn=answer,
    inputs=[gr.Textbox(label="Apna sawaal poochho (Hinglish / English)", lines=2),
            gr.Dropdown(["all", "biology", "chemistry", "physics", "mathematics", "social_science"],
                        value="all", label="Subject"),
            gr.Dropdown(["all", "9", "10"], value="all", label="Class")],
    outputs=[gr.Textbox(label="EduHinglish answer", lines=8), gr.Markdown(label="NCERT sources")],
    title="EduHinglish — NCERT Class 9–10 doubts in Hinglish",
    examples=[[q, s, "all"] for q, s in SAMPLES],
    api_name="answer",
    flagging_mode="never",
)
demo.queue(default_concurrency_limit=1).launch(share=True, debug=True)